In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
df_h1 = session.sql("SELECT * FROM HYPOTHESIS_1_TEST").to_pandas()
df_h2 = session.sql("SELECT * FROM HYPOTHESIS_2_TEST").to_pandas()
df_h3 = session.sql("SELECT * FROM HYPOTHESIS_3_TEST").to_pandas()
df_h4 = session.sql("SELECT * FROM HTEGORY POSSIBLE  n considering that start phase 6 again in very deep n i think all the join operation has to be done in sql as it is not happening in snowflake  SO I CREATED THESE ---HYPOTHESIS_4_TEST").to_pandas()
df_h5 = session.sql("SELECT * FROM HYPOTHESIS_5_ELITE_IMPACT").to_pandas()


In [ ]:
df_h1.groupby("ENGAGEMENT_GROUP")["AVG_RATING"].describe()

In [ ]:
SELECT
  RATING_DOWN_FLAG,
  ENGAGEMENT_DOWN_FLAG,
  COUNT(*) AS CNT
FROM HYPOTHESIS_2_TEST
GROUP BY 1,2;


In [ ]:
-- SQL cell
SELECT
  VOL_GROUP,
  OPENED,
  COUNT(*) AS CNT
FROM HYPOTHESIS_3_TEST
GROUP BY 1,2;


In [ ]:
-- SQL cell
SELECT
  POPULARITY_GROUP,
  OPENED,
  COUNT(*) AS CNT
FROM HYPOTHESIS_4_TEST
GROUP BY 1,2
ORDER BY 1;


In [ ]:
-- SQL cell
SELECT
  REVIEWER_ELITE,
  AVG(AVG_RATING) AS AVG_RATING,
  AVG(RATING_VOLATILITY) AS AVG_VOL,
  COUNT(*) AS CNT
FROM HYPOTHESIS_5_ELITE_IMPACT
GROUP BY 1;


In [ ]:
df_h1["AVG_RATING"].hist(bins=30)
plt.title("Overall Rating Distribution (H1)")
plt.xlabel("AVG RATING")
plt.ylabel("RESTAURANTS")
plt.show()


In [ ]:
plt.scatter(df_h2["ENGAGEMENT_SCORE"], df_h2["RATING_CHANGE_1M"], alpha=0.3)
plt.title("H2: Engagement Score vs 1M Rating Change")
plt.xlabel("ENGAGEMENT SCORE")
plt.ylabel("RATING CHANGE 1M")
plt.show()

In [ ]:
# convert object flags to numbers 1/0 so tests and scatter work
df_h1["CHECKINS_COUNT"] = df_h1["CHECKINS_COUNT"].astype(int)
df_h2["RATING_DOWN_FLAG"] = df_h2["RATING_DOWN_FLAG"].apply(lambda x: 1 if x in [True, 'TRUE', 'true', 'T', '1'] else 0)
df_h2["ENGAGEMENT_DOWN_FLAG"] = df_h2["ENGAGEMENT_DOWN_FLAG"].apply(lambda x: 1 if x in [True, 'TRUE', 'true', 'T', '1'] else 0)


In [ ]:
# working scatter using only real columns
x = df_h1["CHECKINS_COUNT"]
y = df_h1["AVG_RATING"]

plt.scatter(x, y)
plt.title("CHECKINS vs AVG_RATING")
plt.xlabel("CHECKINS")
plt.ylabel("AVG_RATING")
plt.show()


In [ ]:
df_corr = session.sql("""
SELECT
  m.BUSINESS_ID,
  m.REVIEWS_COUNT,
  m.AVG_RATING,
  m.RATING_VOLATILITY,
  c.CHECKINS_COUNT
FROM ANALYTICS_RESTAURANT_MONTHLY m
LEFT JOIN ANALYTICS_RESTAURANT_CHECKINS_MONTHLY c
  ON m.BUSINESS_ID = c.BUSINESS_ID
 AND m.YEAR_MONTH = c.YEAR_MONTH;
""").to_pandas()

df_corr.columns = [c.upper() for c in df_corr.columns]

# compute correlation
df_corr[["REVIEWS_COUNT","AVG_RATING","RATING_VOLATILITY","CHECKINS_COUNT"]].corr()


In [ ]:
g1 = df_h1[df_h1["ENGAGEMENT_GROUP"]=="ENGAGEMENT_DROP"]["AVG_RATING"]
g2 = df_h1[df_h1["ENGAGEMENT_GROUP"]=="NO_ENGAGEMENT_DROP"]["AVG_RATING"]

t_stat, p_val = stats.ttest_ind(g1, g2, nan_policy="omit")
print("H1 P VALUE:", p_val)


In [ ]:
table = pd.crosstab(df_h2["RATING_DOWN_FLAG"], df_h2["ENGAGEMENT_DOWN_FLAG"])
chi2, p2, _, _ = stats.chi2_contingency(table)
print("H2 P VALUE:", p2)

In [ ]:
table3 = pd.crosstab(df_h3["VOL_GROUP"], df_h3["OPENED"])
chi3, p3, _, _ = stats.chi2_contingency(table3)
print("H3 P VALUE:", p3)

In [ ]:
pop_open = df_h4[df_h4["OPENED"]==1]["MAX_MONTHLY_REVIEWS"]
pop_closed = df_h4[df_h4["OPENED"]==0]["MAX_MONTHLY_REVIEWS"]

u, p4 = stats.mannwhitneyu(pop_open, pop_closed)
print("H4 P VALUE:", p4)


In [ ]:
elite_old = df_h5[df_h5["REVIEWER_AGE_YEARS"]>5]["AVG_RATING"]
elite_new = df_h5[df_h5["REVIEWER_AGE_YEARS"]<=5]["AVG_RATING"]

t5, p5 = stats.ttest_ind(elite_old, elite_new, nan_policy="omit")
print("H5 P VALUE:", p5)

In [ ]:
pop_open = df_h4[df_h4["IS_OPEN"]==1]["REVIEW_COUNT"]
pop_closed = df_h4[df_h4["IS_OPEN"]==0]["REVIEW_COUNT"]

u, p4 = stats.mannwhitneyu(pop_open, pop_closed)
print("H4 P VALUE:", p4)

In [ ]:
pop_open = df_h4[df_h4["IS_OPEN"]==1]["REVIEW_COUNT"]
pop_closed = df_h4[df_h4["IS_OPEN"]==0]["REVIEW_COUNT"]

u, p4 = stats.mannwhitneyu(pop_open, pop_closed)
print("H4 P VALUE:", p4)

In [ ]:
df_monthly.describe()
df_checkins.describe()
df_trend.describe()
df_e.describe()

In [ ]:
df_trend["ENGAGEMENT_MOMENTUM"] = df_e.groupby("BUSINESS_ID")["ENGAGEMENT_SCORE"].diff().fillna(0)
df_trend["ENGAGEMENT_MOMENTUM"].describe()

In [ ]:
pop_open = df_h4[df_h4["IS_OPEN"]==1]["REVIEW_COUNT"]
pop_closed = df_h4[df_h4["IS_OPEN"]==0]["REVIEW_COUNT"]

u, p4 = stats.mannwhitneyu(pop_open, pop_closed)
print("H4 P VALUE:", p4)

In [ ]:
df_all = session.sql("""
SELECT
  m.BUSINESS_ID,
  m.REVIEWS_COUNT,
  m.RATING_VOLATILITY,
  c.CHECKINS_COUNT,
  b.OPENED
FROM ANALYTICS_RESTAURANT_MONTHLY m
JOIN ANALYTICS_RESTAURANT_CHECKINS_MONTHLY c
  ON m.BUSINESS_ID = c.BUSINESS_ID
 AND m.YEAR_MONTH = c.YEAR_MONTH
JOIN STG_YELP_BUSINESS b
  ON m.BUSINESS_ID = b.BUSINESS_ID;
""").to_pandas()

df_all.columns = [c.upper() for c in df_all.columns]

# Select only numeric columns for correlation
numeric_cols = ["REVIEWS_COUNT", "RATING_VOLATILITY", "CHECKINS_COUNT", "OPENED"]
df_corr = df_all[numeric_cols]

df_corr.corr()["OPENED"].sort_values()


In [ ]:
df_all["RATING_VOLATILITY"].describe(percentiles=[0.25,0.5,0.75,0.9,0.95,0.99])

In [ ]:
df_sig = session.sql("SELECT * FROM ANALYTICS_RESTAURANT_SIGNAL_IMPORTANCE").to_pandas()
df_sig.columns = [c.upper() for c in df_sig.columns]

# trend correlation, only numeric
num = ["AVG_RATING","RATING_CHANGE_3M","RATING_VOLATILITY","REVIEWS_CHANGE_1M","CHECKINS_CHANGE_1M","REVIEWS_COUNT","CHECKINS_COUNT"]
df_sig[num].corr()["RATING_CHANGE_3M"].sort_values()


In [ ]:
df_sig["YEAR_MONTH"] = pd.to_datetime(df_sig["YEAR_MONTH"] + "-01")

In [ ]:
df_sig["REVIEWS_DOWN_FLAG"] = df_sig["REVIEWS_CHANGE_1M"].apply(lambda x: 1 if x < 0 else 0)
df_sig["CHECKINS_DOWN_FLAG"] = df_sig["CHECKINS_CHANGE_1M"].apply(lambda x: 1 if x < 0 else 0)
df_sig["VOLATILITY_HIGH_FLAG"] = df_sig["RATING_VOLATILITY"].apply(
    lambda x: 1 if x > df_sig["RATING_VOLATILITY"].median() else 0
)


In [ ]:
numeric_cols = [
    "REVIEWS_COUNT",
    "CHECKINS_COUNT",
    "AVG_RATING",
    "RATING_VOLATILITY",
    "REVIEWS_CHANGE_1M",
    "CHECKINS_CHANGE_1M",
    "REVIEWS_DOWN_FLAG",
    "CHECKINS_DOWN_FLAG",
    "VOLATILITY_HIGH_FLAG",
    "OPENED"
]

df_num = df_sig[numeric_cols]
df_num.corr()["OPENED"].sort_values()


In [ ]:
df_trend = session.sql("""
    SELECT BUSINESS_ID, YEAR_MONTH, RATING_CHANGE_3M
    FROM ANALYTICS_RESTAURANT_RISK_FINAL;
""").to_pandas()

df_trend.columns = [c.upper() for c in df_trend.columns]
df_trend["YEAR_MONTH"] = pd.to_datetime(df_trend["YEAR_MONTH"] + "-01")

df_sig = df_sig.merge(df_trend, on=["BUSINESS_ID","YEAR_MONTH"], how="inner")

# then cluster with it
cluster_features = [
    "REVIEWS_CHANGE_1M",
    "CHECKINS_CHANGE_1M",
    "RATING_VOLATILITY",
    "RATING_CHANGE_3M"
]

df_cluster = df_sig[cluster_features].fillna(0)

kmeans = KMeans(n_clusters=5, random_state=42)
kmeans.fit(df_cluster)

df_sig["RISK_CLUSTER"] = kmeans.labels_
df_sig["RISK_CLUSTER"].value_counts()


In [ ]:
df_sig.groupby("YEAR_MONTH")["REVIEWS_COUNT"].sum().plot()
plt.title("TOTAL REVIEWS TREND")
plt.xlabel("MONTH")
plt.ylabel("REVIEWS")
plt.show()


In [ ]:
df_sig.groupby("YEAR_MONTH")["CHECKINS_COUNT"].sum().plot()
plt.title("TOTAL CHECKINS TREND")
plt.xlabel("MONTH")
plt.ylabel("CHECKINS")
plt.show()


In [ ]:
df_sig["RATING_VOLATILITY"].hist(bins=50)
plt.title("VOLATILITY DISTRIBUTION")
plt.xlabel("VOLATILITY")
plt.ylabel("RESTAURANTS")
plt.show()


In [ ]:
plt.scatter(df_sig["REVIEWS_CHANGE_1M"], df_sig["RATING_CHANGE_3M"])
plt.title("ENGAGEMENT CHANGE VS RATING FALL")
plt.xlabel("REVIEW CHANGE 1M")
plt.ylabel("RATING CHANGE 3M")
plt.show()


In [ ]:
df_sig.groupby("YEAR_MONTH")["CHECKINS_COUNT"].sum().plot()
plt.title("TOTAL CHECKINS TREND")
plt.xlabel("MONTH")
plt.ylabel("CHECKINS")
plt.show()


In [ ]:
df_sig["RATING_VOLATILITY"].hist(bins=50)
plt.title("VOLATILITY DISTRIBUTION")
plt.xlabel("VOLATILITY")
plt.ylabel("RESTAURANTS")
plt.show()


In [ ]:
plt.scatter(df_sig["REVIEWS_CHANGE_1M"], df_sig["RATING_CHANGE_3M"])
plt.title("ENGAGEMENT CHANGE VS RATING FALL")
plt.xlabel("REVIEW CHANGE 1M")
plt.ylabel("RATING CHANGE 3M")
plt.show()


In [ ]:
# Cohen's D for rating difference (H1 effect size)
mean1 = group1.mean()
mean2 = group2.mean()
std1 = group1.std()
std2 = group2.std()
pooled_std = ((std1**2 + std2**2)/2)**0.5
cohen_d = (mean1 - mean2)/pooled_std
print("COHEN D EFFECT SIZE H1:", cohen_d)


In [ ]:
SELECT
  VOL_GROUP,
  OPENED,
  COUNT(*) AS CNT,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY VOL_GROUP), 2) AS CLOSURE_PERCENT
FROM HYPOTHESIS_3_TEST
GROUP BY 1,2
ORDER BY 1,2;

In [ ]:
SELECT
  VOL_GROUP,
  OPENED,
  COUNT(*) AS CNT,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY VOL_GROUP), 2) AS CLOSURE_PERCENT
FROM HYPOTHESIS_3_TEST
GROUP BY 1,2
ORDER BY 1,2;


In [ ]:
SELECT
  ENGAGEMENT_GROUP,
  OPENED,
  COUNT(*) AS CNT,
  ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (PARTITION BY ENGAGEMENT_GROUP), 2) AS CLOSURE_PERCENT
FROM HYPOTHESIS_1_TEST
GROUP BY 1,2
ORDER BY 1,2;


In [ ]:
from sklearn.metrics import roc_auc_score
df = session.sql("SELECT * FROM ANALYTICS_TARGET_STRESS").to_pandas()
df.columns = [c.upper() for c in df.columns]
df["RISK_SCORE"] = df_m["REVIEWS_COUNT"].fillna(0)  # temp for testing pipeline

print("NEW AUROC:", roc_auc_score(df["RATING_STRESS_FLAG"], df["RISK_SCORE"]))


In [ ]:
df = session.sql("SELECT * FROM ANALYTICS_RISK_SIGNALS_NORMALIZED").to_pandas()
df.columns = [c.upper() for c in df.columns]
df["RISK_SIGNAL_SCORE"] = df["RISK_SIGNAL_SCORE"].fillna(0)

from sklearn.metrics import roc_auc_score
print("AUROC:", roc_auc_score(df["IS_OPEN"], df["RISK_SIGNAL_SCORE"]))


In [ ]:
CREATE OR REPLACE TABLE CATEGORY_ENCODING_BUSINESS AS
WITH CTE AS (
  SELECT
    BUSINESS_ID,
    TRIM(VALUE)::STRING AS CATEGORY
  FROM STG_YELP_BUSINESS,
  LATERAL SPLIT_TO_TABLE(CATEGORIES, ',')
)
SELECT BUSINESS_ID, CATEGORY
FROM CTE;


In [ ]:
# Load again from SQL (fresh notebook)
df_sig = session.sql("SELECT * FROM ANALYTICS_RESTAURANT_SIGNAL_IMPORTANCE").to_pandas()
df_target = session.sql("SELECT BUSINESS_ID, YEAR_MONTH, RATING_STRESS_FLAG FROM ANALYTICS_TARGET_STRESS").to_pandas()

# Capitalize columns
df_sig.columns = [c.upper() for c in df_sig.columns]
df_target.columns = [c.upper() for c in df_target.columns]

# Convert YEAR_MONTH to date
df_sig["YEAR_MONTH"] = pd.to_datetime(df_sig["YEAR_MONTH"] + "-01")
df_target["YEAR_MONTH"] = pd.to_datetime(df_target["YEAR_MONTH"] + "-01")

# Join properly
df_train = df_sig.merge(df_target, on=["BUSINESS_ID","YEAR_MONTH"], how="inner")

# Check if rows exist now
print("TOTAL ROWS AFTER JOIN:", len(df_train))


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

X = df_train[[
    "REVIEWS_COUNT",
    "CHECKINS_COUNT",
    "RATING_VOLATILITY"
]].fillna(0)

y = df_train["RATING_STRESS_FLAG"].astype(int)

if len(df_train) == 0:
    print("NO DATA – MODEL CANNOT RUN")
else:
    model = LogisticRegression(max_iter=1000)
    model.fit(X, y)
    pred = model.predict_proba(X)[:, 1]
    print("TRAINED MODEL AUROC:", roc_auc_score(y, pred))


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

X = df_train[["REVIEWS_COUNT","CHECKINS_COUNT","RATING_VOLATILITY","RATING_CHANGE_3M"]]
y = df_train["RATING_STRESS_FLAG"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=200)
model.fit(X_train, y_train)

pred = model.predict_proba(X_test)[:,1]
print("RF AUROC TEST:", roc_auc_score(y_test, pred))


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd

FEATURES = [
    "REVIEWS_COUNT",
    "CHECKINS_COUNT",
    "RATING_VOLATILITY"
]

TARGET = "RATING_STRESS_FLAG"

X = df_train[FEATURES].fillna(0)
y = df_train[TARGET].fillna(0).astype(int)

X_TRAIN, X_TEST, Y_TRAIN, Y_TEST = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_TRAIN, Y_TRAIN)

pred = model.predict_proba(X_TEST)[:, 1]
score = roc_auc_score(Y_TEST, pred)

print("UPDATED AUROC (NO LEAKAGE):", score)


In [ ]:
CREATE OR REPLACE TABLE ANALYTICS_TARGET AS
SELECT BUSINESS_ID,
       STARS,
       REVIEW_COUNT,
       IS_OPEN AS CLOSED_FLAG
FROM STG_YELP_BUSINESS;


In [ ]:
CREATE OR REPLACE TABLE ANALYTICS_STRESS_TARGET AS
SELECT
  BUSINESS_ID,
  YEAR_MONTH,
  AVG_RATING,
  LAG(AVG_RATING,3) OVER (PARTITION BY BUSINESS_ID ORDER BY TO_DATE(YEAR_MONTH||'-01')) AS RATING_3M_AGO,
  ROUND(AVG_RATING - RATING_3M_AGO,2) AS RATING_CHANGE_3M,
  CASE WHEN RATING_CHANGE_3M < -0.4 THEN 1 ELSE 0 END AS RATING_STRESS_FLAG
FROM ANALYTICS_RESTAURANT_MONTHLY;


In [ ]:
SELECT RATING_STRESS_FLAG, COUNT(*) FROM ANALYTICS_STRESS_TARGET GROUP BY 1;

In [ ]:
CREATE OR REPLACE TABLE ANALYTICS_FINAL_FEATURES AS
WITH REV AS (
  SELECT BUSINESS_ID,
         TO_CHAR(DATE_TRUNC('MONTH', REVIEW_DATE),'YYYY-MM') AS YEAR_MONTH,
         COUNT(*) AS REVIEW_VOL,
         AVG(REVIEW_STARS) AS AVG_RATING,
         STDDEV(REVIEW_STARS) AS RATING_VOLATILITY,
         SUM(SENTIMENT_SCORE) AS TOTAL_SENTIMENT,
         SUM(CASE WHEN SENTIMENTS='NEGATIVE' THEN 1 ELSE 0 END) AS NEG_VOL,
         SUM(CASE WHEN SENTIMENTS='POSITIVE' THEN 1 ELSE 0 END) AS POS_VOL
  FROM STG_YELP_REVIEWS
  GROUP BY 1,2
),
CHK AS (
  SELECT BUSINESS_ID,
         TO_CHAR(DATE_TRUNC('MONTH', DATE_VISITED),'YYYY-MM') AS YEAR_MONTH,
         COUNT(*) AS CHECKIN_VOL
  FROM STG_TBL_CHECKINS
  GROUP BY 1,2
),
USR AS (
  SELECT USER_ID,
         REVIEW_COUNT AS USER_TOTAL_REVIEWS,
         ELITE,
         DATEDIFF('YEAR', YELPING_SINCE, CURRENT_DATE()) AS USER_AGE_YEARS
  FROM STG_YELP_USER
)
SELECT
  R.BUSINESS_ID,
  R.YEAR_MONTH,
  COALESCE(C.CHECKIN_VOL,0) AS CHECKINS_THIS_MONTH,
  R.REVIEW_VOL AS REVIEWS_THIS_MONTH,
  R.AVG_RATING,
  COALESCE(R.RATING_VOLATILITY,0) AS RATING_VOLATILITY,
  ROUND(R.NEG_VOL / NULLIF(R.REVIEW_VOL,0),3) AS NEG_SENTIMENT_RATIO,
  ROUND(R.POS_VOL / NULLIF(R.REVIEW_VOL,0),3) AS POS_SENTIMENT_RATIO,
  USR.USER_TOTAL_REVIEWS,
  CASE WHEN USR.ELITE IS NOT NULL AND USR.ELITE <> '' THEN 1 ELSE 0 END AS ELITE_REVIEWER_FLAG,
  B.IS_OPEN AS OPENED,
  B.REVIEW_COUNT AS TOTAL_BUSINESS_REVIEWS,
  B.STARS AS BUSINESS_STARS,
  B.CATEGORIES AS CATEGORY
FROM REV R
LEFT JOIN CHK C
  ON R.BUSINESS_ID = C.BUSINESS_ID
 AND R.YEAR_MONTH = C.YEAR_MONTH
JOIN STG_YELP_BUSINESS B
  ON R.BUSINESS_ID = B.BUSINESS_ID
LEFT JOIN USR USR
  ON R.USER_ID = USR.USER_ID
ORDER BY R.BUSINESS_ID, R.YEAR_MONTH;


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, classification_report, accuracy_score, precision_score, recall_score, f1_score

# --- LOAD FEATURE & TARGET TABLES ---
df_feat = session.sql("SELECT * FROM ANALYTICS_FINAL_FEATURES").to_pandas()
df_target = session.sql("SELECT BUSINESS_ID, YEAR_MONTH, RATING_STRESS_FLAG FROM ANALYTICS_STRESS_TARGET").to_pandas()

# --- CAPITALIZE COLUMN NAMES ---
df_feat.columns = [c.upper() for c in df_feat.columns]
df_target.columns = [c.upper() for c in df_target.columns]

# --- CONVERT YEAR_MONTH TO REAL DATE FOR SAFE JOIN ---
df_feat["YEAR_MONTH"] = pd.to_datetime(df_feat["YEAR_MONTH"] + "-01")
df_target["YEAR_MONTH"] = pd.to_datetime(df_target["YEAR_MONTH"] + "-01")

# --- JOIN FEATURES WITH TARGET ---
df = df_feat.merge(df_target, on=["BUSINESS_ID", "YEAR_MONTH"], how="inner")

print("ROWS AFTER JOIN:", len(df))
if len(df) == 0:
    raise ValueError("JOIN RETURNED 0 ROWS — DATA PIPELINE ISSUE")

# --- ENCODE CATEGORY (NOMINAL ENCODING) ---
df["CATEGORY_ENC"] = LabelEncoder().fit_transform(df["CATEGORY"].fillna("UNKNOWN"))

# --- SCALE NUMERIC FEATURES (SENIOR BEST PRACTICE) ---
NUMERIC = [
    "REVIEWS_THIS_MONTH",
    "CHECKINS_THIS_MONTH",
    "AVG_RATING",
    "RATING_VOLATILITY",
    "NEG_SENTIMENT_RATIO",
    "POS_SENTIMENT_RATIO",
    "TOTAL_BUSINESS_REVIEWS",
    "BUSINESS_STARS"
]

scaler = StandardScaler()
df[NUMERIC] = scaler.fit_transform(df[NUMERIC].fillna(0))

# --- FINAL MODEL FEATURES (NO TARGET LEAK) ---
FEATURES = NUMERIC + ["CATEGORY_ENC", "ELITE_REVIEWER_FLAG"]

X = df[FEATURES]
y = df["RATING_STRESS_FLAG"].astype(int)

# --- CHECK CLASS BALANCE BEFORE TRAINING ---
print("\nCLASS BALANCE:\n", y.value_counts())

# --- SPLIT INTO UNSEEN TRAIN/TEST ---
X_TRAIN, X_TEST, Y_TRAIN, Y_TEST = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# --- TRAIN RANDOM FOREST WITH BALANCED WEIGHTS ---
model = RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=42)
model.fit(X_TRAIN, Y_TRAIN)

# --- PREDICT ON TEST ---
pred_prob = model.predict_proba(X_TEST)[:, 1]
pred_class = model.predict(X_TEST)

# --- VALIDATION METRICS ---
auroc = roc_auc_score(Y_TEST, pred_prob)
acc = accuracy_score(Y_TEST, pred_class)
prec = precision_score(Y_TEST, pred_class)
rec = recall_score(Y_TEST, pred_class)
f1 = f1_score(Y_TEST, pred_class)

print("\nMODEL TEST PERFORMANCE:")
print("AUROC:", auroc)
print("ACCURACY:", acc)
print("PRECISION:", prec)
print("RECALL:", rec)
print("F1 SCORE:", f1)

print("\nDETAILED REPORT:\n", classification_report(Y_TEST, pred_class))


In [ ]:
CREATE OR REPLACE VIEW ANALYTICS_MOMENTUM_FEATURES AS
WITH MONTHLY_BASE AS (
    -- Your existing monthly aggregation
    SELECT 
        BUSINESS_ID,
        YEAR_MONTH,
        AVG(STARS_GIVEN) as AVG_RATING,
        COUNT(REVIEW_ID) as REVIEW_VOL,
        SUM(CASE WHEN SENTIMENT = 'NEGATIVE' THEN 1 ELSE 0 END) / NULLIF(COUNT(REVIEW_ID), 0) as NEG_RATIO
    FROM STG_YELP_REVIEWS
    GROUP BY 1, 2
)
SELECT 
    *,
    -- 1. RATING VELOCITY: 3-month slope (Is quality dropping?)
    AVG_RATING - LAG(AVG_RATING, 3) OVER (PARTITION BY BUSINESS_ID ORDER BY YEAR_MONTH) AS RATING_VELOCITY_3M,

    -- 2. ENGAGEMENT DROP: Current reviews vs 6-month average (Is interest fading?)
    REVIEW_VOL / NULLIF(AVG(REVIEW_VOL) OVER (PARTITION BY BUSINESS_ID ORDER BY YEAR_MONTH ROWS BETWEEN 6 PRECEDING AND 1 PRECEDING), 0) AS ENGAGEMENT_MOMENTUM,

    -- 3. SENTIMENT SHOCK: Did negative sentiment spike above its own baseline?
    NEG_RATIO - AVG(NEG_RATIO) OVER (PARTITION BY BUSINESS_ID ORDER BY YEAR_MONTH ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS SENTIMENT_SURGE_ABS
FROM MONTHLY_BASE;

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# 1. Separate Features
NUMERIC_FEATURES = NUMERIC # From your previous list
CATEGORICAL_FEATURES = ["CATEGORY"]

# 2. Define Preprocessing Stages
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')), # Median is safer than 0 for risk
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='UNKNOWN')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')) # Handles new categories in test data
])

# 3. Combine into a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, NUMERIC_FEATURES),
        ('cat', categorical_transformer, CATEGORICAL_FEATURES)
    ]
)

# 4. Final System (The "Senior" Way)
clf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=500, class_weight="balanced_subsample", random_state=42))
])

# Now, split and fit properly
X_TRAIN, X_TEST, Y_TRAIN, Y_TEST = train_test_split(df[FEATURES], y, test_size=0.3, stratify=y)
clf.fit(X_TRAIN, Y_TRAIN)

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import PrecisionRecallDisplay

# Show the Trade-off
display = PrecisionRecallDisplay.from_estimator(clf, X_TEST, Y_TEST)
plt.title("Precision-Recall Curve: Essential for High-Stakes Risk")
plt.show()